# Day 6: Publish your agent

You already have: **ingest → search → agent → evaluation**. The agent still lives in notebooks. Today you **package** it, add a **web UI**, and optionally **deploy** it.

You will:

- **Refactor** into clean Python modules under **`course/app/`**
- Run a **CLI** (`main.py`) and a **Streamlit** app (`app.py`)
- **Deploy** (e.g. Streamlit Community Cloud) with `OPENAI_API_KEY` in secrets

**Needs:** `OPENAI_API_KEY` (e.g. `ai_hero/.env`).


## 1. Clean layout

Instead of one giant notebook, the runnable app lives in **`course/app/`**:

| File | Role |
|------|------|
| `ingest.py` | Download repo zip, optional **chunking**, **`Index.fit`** |
| `search_tools.py` | **`SearchTool`** class wrapping `index.search` |
| `search_agent.py` | **`Agent`** + system prompt template (GitHub blob links) |
| `logs.py` | JSON logs under `LOGS_DIRECTORY` or `./logs` |
| `main.py` | **CLI**: `asyncio.run(agent.run(...))` |
| `app.py` | **Streamlit** chat + **streaming** via `run_stream_sync` + `stream_text` |

Improvements vs. a single notebook:

- **`read_repo_data`** strips the zip root (`split('/', maxsplit=1)`) so **filenames** are stable for citations.
- **`index_data(..., doc_filter=...)`** keeps the **data-engineering** FAQ filter in one place.
- Tools are **`tools=[search_tool.search]`** (bound method), not a closure over a global index.


## 2. Install & run (local)

```bash
cd course/app
uv sync
```

CLI (downloads & indexes on startup):

```bash
uv run python main.py
```

Streamlit:

```bash
uv run streamlit run app.py
```

Open the printed URL. First load builds the index (can take a minute).


## 3. Streaming (Pydantic AI ≥ 1.7)

The UI uses **`agent.run_stream_sync(user_prompt=...)`** and **`result.stream_text(delta=True)`** so tokens appear incrementally. After the stream finishes, **`logs.log_interaction_to_file(agent, result.new_messages())`** runs once.

Optional: pass **`message_history=...`** into `run` / `run_stream_sync` if you want multi-turn memory in the app.


## 4. Deploy (Streamlit Cloud)

1. Push this repo to GitHub.
2. Create an app pointing at **`course/app`** with main file **`app.py`** (adjust UI paths if your repo root differs).
3. In **Secrets**, add:

   ```toml
   OPENAI_API_KEY = "sk-..."
   ```

4. If **`uv`** is not used on the host, export a lockfile:

   ```bash
   cd course/app
   uv export --no-dev --no-hashes -o requirements.txt
   ```

   Commit **`requirements.txt`** so the platform can `pip install -r requirements.txt`.

5. Optional: set **`LOGS_DIRECTORY`** to a writable path (e.g. `/tmp/logs`) if the default is read-only.

Treat public demos as **billable**: rate-limit, cap usage, or use a dedicated API key.


In [ ]:
# Quick sanity check: list app files from the course folder (run with cwd = course/ or adjust path)
from pathlib import Path

app_dir = Path("app")
if app_dir.is_dir():
    for p in sorted(app_dir.glob("*.py")):
        print(p)
else:
    print("Run this cell with working directory set to the course/ folder, or open files in course/app/ in the editor.")


## Homework (course)

- Skim each module in **`course/app/`** and trace one request: **UI → agent → tool → index → log**.
- Deploy to Streamlit Cloud **or** document why you skipped it (keys, cost).
- Share progress: [course](https://alexeygrigorev.com/aihero/), DataTalks `#course-ai-hero`.
